# Live Demo — WRDS Data Pull
### Session 5 · ExInt II · WU Vienna · SS 2026

**What we do in this notebook:**
1. Connect to WRDS
2. Explore what Compustat Global looks like
3. Pull 5 firms for fiscal years 2015–2024 — all variables
4. Inspect the data
5. Save as parquet in a timestamped folder

This is the **mini version** of `01_pull_data.py` — same logic, small data so we can see results instantly.

> **Reference project:** https://github.com/vkiefner/sme-intl

---
## Cell 1 — Imports & credentials

In [ ]:
import os
from datetime import datetime
from pathlib import Path

import pandas as pd
import wrds
from dotenv import load_dotenv

# Load WRDS username from .env
load_dotenv()
WRDS_USER = os.getenv("WRDS_USERNAME")

print(f"WRDS user: {WRDS_USER}")
print(f"pandas:    {pd.__version__}")

---
## Cell 2 — Connect to WRDS

In [ ]:
db = wrds.Connection(wrds_username=WRDS_USER)
print("Connected!")

---
## Cell 3 — What tables are available in Compustat Global?

In [ ]:
# List all tables in the Compustat Global library
tables = db.list_tables(library="comp_global_daily")
print(f"{len(tables)} tables available:")
for t in tables:
    print(" ", t)

---
## Cell 4 — What columns does g_funda have?

In [ ]:
# g_funda = Global Fundamentals Annual — the main firm-year table
schema = db.describe_table(library="comp_global_daily", table="g_funda")
print(f"{len(schema)} columns available in g_funda")
schema.head(20)

---
## Cell 5 — Choose 5 firms to pull

We use `gvkey` — the Compustat firm identifier.  
These are 5 well-known European firms spanning different countries and sectors.

| gvkey  | Firm     | Country |
|--------|----------|---------|
| 005073 | Siemens  | DEU     |
| 012141 | Nestlé   | CHE     |
| 101407 | Airbus   | FRA     |
| 012179 | Philips  | NLD     |
| 061411 | Zumtobel | AUT     |

In [ ]:
# Five firms — feel free to swap any of these for firms relevant to your project
FIVE_FIRMS = ('005073', '012141', '101407', '012179', '061411')

# Build the SQL IN-clause string
firms_sql = ", ".join(f"'{g}'" for g in FIVE_FIRMS)
print(f"Pulling gvkeys: {firms_sql}")

---
## Cell 6 — Pull ALL variables for these 5 firms, 2015–2024

In [ ]:
query = f"""
    SELECT *
    FROM comp_global_daily.g_funda
    WHERE gvkey   IN ({firms_sql})
      AND fyear   BETWEEN 2015 AND 2024
      AND indfmt  = 'INDL'
      AND datafmt = 'STD'
      AND popsrc  = 'I'
      AND consol  = 'C'
    ORDER BY gvkey, fyear
"""

print("Pulling from WRDS...")
df_raw = db.raw_sql(query, date_cols=["datadate"])

print(f"\nDone! Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

---
## Cell 7 — First look at the data

In [ ]:
# First 5 rows
df_raw.head()

In [ ]:
# Column names — all available variables
print(f"Total columns: {len(df_raw.columns)}")
print("\nAll column names:")
for i, col in enumerate(df_raw.columns):
    print(f"  {i:>3}. {col}")

In [ ]:
# Data types
df_raw.dtypes

In [ ]:
# Which firms did we get? How many years each?
df_raw.groupby(["gvkey", "conm"])["fyear"].agg(["min", "max", "count"]).rename(
    columns={"min": "first_year", "max": "last_year", "count": "n_years"}
)

---
## Cell 8 — Look at key financial variables

In [ ]:
# Select the most commonly used variables for a quick preview
key_vars = ["gvkey", "conm", "fyear", "loc",
            "at",    # total assets
            "sale",  # net sales
            "ib",    # income before extraordinary items (net income)
            "xrd",   # R&D expenditure
            "emp",   # employees (thousands)
            "dltt",  # long-term debt
            "pifo",  # pre-tax income from foreign operations
            "curcd" # currency
           ]

# Keep only columns that exist in our pull
available = [v for v in key_vars if v in df_raw.columns]
df_raw[available]

In [ ]:
# Quick summary stats on numeric columns
df_raw[["at", "sale", "ib", "xrd", "emp"]].describe().round(2)

---
## Cell 9 — How many missing values per column?

In [ ]:
# Missing value count and percentage
missing = pd.DataFrame({
    "missing_n":   df_raw.isnull().sum(),
    "missing_pct": (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
})

# Show only columns that have at least one missing value
missing[missing["missing_n"] > 0].sort_values("missing_pct", ascending=False)

---
## Cell 10 — Standardize column names

In [ ]:
# Lowercase + strip whitespace — same as the full pull script
df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

print("Column names standardized.")
print("Before:", list(df_raw.columns[:5]))
print("After: ", list(df.columns[:5]))

---
## Cell 11 — Save to timestamped parquet folder

Same structure as `01_pull_data.py`: each run gets its own folder named with the current date and time.

In [ ]:
# Create timestamped output folder
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
out_dir   = Path("data") / "raw" / timestamp
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Output folder: {out_dir}")

# Save one parquet file per year (same chunking strategy as full pull)
for year, group in df.groupby("fyear"):
    out_path = out_dir / f"fyear_{int(year)}.parquet"
    group.to_parquet(out_path, index=False)
    print(f"  fyear_{int(year)}.parquet  →  {len(group)} rows")

# Write metadata
(out_dir / "pull_metadata.txt").write_text(
    f"LIVE DEMO PULL (5 firms)\n"
    f"========================\n"
    f"Pulled:       {datetime.now().isoformat()}\n"
    f"Source:       comp_global_daily.g_funda\n"
    f"Firms:        {', '.join(FIVE_FIRMS)}\n"
    f"Fiscal years: 2015–2024\n"
    f"Rows:         {len(df)}\n"
    f"Columns:      {df.shape[1]}\n"
)

print(f"\nDone! {len(df)} rows saved across {df['fyear'].nunique()} year files.")

---
## Cell 12 — Verify: read back one parquet file

In [ ]:
# Read back the 2024 file to confirm it saved correctly
check = pd.read_parquet(out_dir / "fyear_2024.parquet")
print(f"fyear_2024.parquet: {check.shape[0]} rows × {check.shape[1]} columns")
check[["gvkey", "conm", "fyear", "at", "sale", "ib"]].head()

---
## Cell 13 — Close the connection

In [ ]:
db.close()
print("WRDS connection closed.")
print(f"\nYour data is in: {out_dir}")
print(f"Next step:       run 02_clean.py (or the full 01_pull_data.py for your own project)")

---
## Summary — What just happened

| Step | What we did |
|------|-------------|
| Connected | `wrds.Connection()` using credentials from `.env` |
| Explored | Listed tables and columns available in Compustat Global |
| Queried | `SELECT *` for 5 firms, 2015–2024, with standard filters |
| Inspected | Shape, column names, dtypes, missing values |
| Cleaned names | Lowercase column names |
| Saved | Parquet files in a timestamped folder under `data/raw/` |

**Your full script `01_pull_data.py` does exactly the same thing** — just for ALL firms globally instead of 5, and chunked by fiscal year to handle the larger volume.

---
*ExInt II · WU Vienna · SS 2026 · github.com/vkiefner/sme-intl*